# ClimX Playground

This notebook provides a end-to-end pipeline for model development. 

In [ ]:
#Autoreload .py files
%load_ext autoreload
%autoreload 2

#https://github.com/chmp/ipytest/issues/80
import sys
sys.breakpointhook = sys.__breakpointhook__

Uset the `setup_logger` function to set the logging level to `INFO` or `DEBUG` at any time during the notebook.

In [ ]:
from src.utils.logging_utils import setup_logger, logging
setup_logger(logging.INFO)

# Data loading and preprocessing

Data can be found at hugging face link 

In [ ]:
# Import necessary libraries
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np


# Set plot style
plt.style.use('ggplot')
sns.set_context("notebook", font_scale=1.2)

# Configure plots to be larger
plt.rcParams['figure.figsize'] = [12, 8]

# Display all rows and columns of xarray datasets
xr.set_options(display_width=200, display_max_rows=200)

# Define data version
DATA_VERSION = 'lite' # Options: 'lite', 'full'
stationarization_mode = 'monthly'

# Define variables configuration
TARGET_VARIABLES = ['huss', 'pr', 'psl', 'sfcWind', 'tas', 'tasmax', 'tasmin']
GREENHOUSE_GAS_AGENTS = ['CO2_LBC', 'N2O_LBC', 'CH4_LBC', 'CFC11eq_LBC', 'CF2CL2_LBC']
AEROSOL_AGENTS = ['BC_AX', 'BC_N', 'SO2', 'SO4_PR', 'OM_NI']
ALL_FORCING_VARS = GREENHOUSE_GAS_AGENTS + AEROSOL_AGENTS

# Path to the data
output_path = Path('/data/users/climate_challenge/final/')
data_path = Path('data/')
forcing_data_path = Path('/data/users/climate_challenge/data/forcing/inputs/')
nc_data_path = data_path / 'ncfiles'

In [ ]:
# --- Ensure data exists locally; otherwise download from Hugging Face ---
import os
from pathlib import Path
from src.utils.hugging_face_utils import get_dataset_from_hf

# Download into data_path if not already present
get_dataset_from_hf(data_path, variant=DATA_VERSION)

In [ ]:
# --- Load the dataset --- 
# if RuntimeError: NetCDF: HDF error, just re-run this cell and it should work 
from src.utils.hugging_face_utils import open_climx_virtual_datasets
dataset = open_climx_virtual_datasets(data_path, DATA_VERSION)

In [ ]:
# --- Preprocess the data (id needed) ---
from src.data_preprocessing.preprocessing import preprocess_train

preprocessing_path = Path(f"preprocessing_data_{DATA_VERSION}/") 
precomputed = True
save_preprocessed = True

# Load and preprocess the training data
if not precomputed:
    X_train, y_train, metadata = preprocess_train(
        dataset, 
        preprocessing_path=preprocessing_path,
        version='lite',
        scaling_params_path=None, #'preprocessing_data_lite/scaling_params_lite.json'
    )
    if save_preprocessed:
        X_train = X_train.chunk({'time': 30, 'lat': -1, 'lon': -1})
        X_train.to_netcdf(f'preprocessing_data_{DATA_VERSION}/X_train.nc')
        y_train = y_train.chunk({'time': 30, 'lat': -1, 'lon': -1})
        y_train.to_netcdf(f'preprocessing_data_{DATA_VERSION}/y_train.nc')
else:
    X_train = xr.open_dataset(preprocessing_path / 'X_train.nc')
    y_train = xr.open_dataset(preprocessing_path / 'y_train.nc')

# Training 

In [ ]:
# Define model to train
model_type = 'gnn' # implemented options in models/: climatology, nn, gnn 

# Instantiate the appropriate model
if model_type == 'climatology':
    from src.models.climatology_model import ClimatologyBaseline
    emulator = ClimatologyBaseline()
    model_path = Path(f'models_{DATA_VERSION}') / f'{model_type}.nc'
    trainer_params = {} # No training parameters needed for climatology model
    
elif model_type == 'nn_v2':
    from src.models.nn_model import NNBaseline
    import torch
    torch.manual_seed(42)  # For reproducibility
    # NN model requires parameters at initialization
    model_params = {
        'n_lat': 192, # This should match the subsampled data
        'n_lon': 288, # This should match the subsampled data
        'n_forcing_vars': 22,
        'n_target_vars': 7,
        'learning_rate': 1e-3,
        'separate_output_heads': True, # Whether to have separate output heads for each variable
    }
    model_path = Path(f'models_{DATA_VERSION}') / f'{model_type}'
    trainer_params = {'max_epochs': 5, 'model_path': model_path} 

    emulator = NNBaseline(model_params)

elif model_type == 'gnn':
    from src.models.gnn_model import GNNBaseline
    import torch
    torch.manual_seed(42)  # For reproducibility
    # NN model requires parameters at initialization
    folder = "src/models/adjacency/"
    graph_size = DATA_VERSION+"_" if DATA_VERSION == 'lite' else ''
    graph = 'random0.000001'  # Options: 'spatial', "spatial_0.000001", 'random0.000001', 'random0.00001'
    edge_index = torch.load(f"{folder}{graph_size}{graph}_edge_index.pt")
    model_params = {
        'edge_index': edge_index,
        'n_forcing_vars': 22,
        'n_target_vars': 7,
        'learning_rate': 1e-3,
        #'separate_output_heads': True, # Whether to have separate output heads for each variable
    }
    model_path = Path(f'models_{DATA_VERSION}') / f'{model_type}_{graph}'
    trainer_params = {'max_epochs': 5, 'model_path': model_path} 

    emulator = GNNBaseline(model_params)
else:
    raise ValueError(f"Unknown model type: {model_type}")

print(f"Using '{model_type}' model. Model will be saved to: {model_path}")

In [ ]:
# Fit the model
FIT_MODEL = False

# Fit and save the model
if FIT_MODEL:
    emulator.fit(X_train, y_train, trainer_params=trainer_params)
    emulator.save(model_path)
    print(f"'{model_type}' model trained and saved to {model_path}")
else:
    # Load the model
    emulator.load(model_path / 'last.ckpt')
    print(f"'{model_type}' model loaded from {model_path}")

# Plot the loss
if hasattr(emulator, 'plot_loss'):
    emulator.plot_loss()

# Evaluation

In [ ]:
# --- Evaluate the selected model ---
from src.data_preprocessing.preprocessing import preprocess_test

precomputed = True
save_preprocessed = True

# Load and preprocess the test data
if not precomputed:
    X_test, metadata_test = preprocess_test(
        dataset, 
        preprocessing_path=preprocessing_path,
        version='lite',
        scaling_params_path=preprocessing_path / f'scaling_parameters_{DATA_VERSION}.json',
    )
    if save_preprocessed:
        X_test = X_test.astype(np.float32).chunk({'time': 30, 'lat': -1, 'lon': -1})
        print("Saving X_test...")
        encoding = {var: {'zlib': False, '_FillValue': None} for var in X_test.data_vars}

        from dask.diagnostics import ProgressBar

        X_test.to_netcdf(preprocessing_path / "X_test.nc", encoding=encoding, compute=False)
       
else:
    import json 

    print("Loading preprocessed test data ...")
    X_test = xr.open_dataset(preprocessing_path / 'X_test.nc')
    metadata_test = {
        'climatology_path': str(preprocessing_path / f'climatology_{DATA_VERSION}.nc'),
        'scaling_params_path': str(preprocessing_path / f'scaling_parameters_{DATA_VERSION}.json'),
        'scaling_params': json.load(open(preprocessing_path / f'scaling_parameters_{DATA_VERSION}.json')),
        'stationarization_mode': stationarization_mode,
    }


# Define paths for evaluation results
results_path = Path(f'./results_{DATA_VERSION}/{model_type}_evaluation.json')
predictions_path = Path(f'./results_{DATA_VERSION}/{model_type}_predictions.zarr')

# Evaluate the model
y_pred, indices_pred = emulator.evaluate(
    X_test, 
    metadata_test, 
    results_path, 
    predictions_path,
    dataset.hist, 
    TARGET_VARIABLES=TARGET_VARIABLES,
    LOAD_PREDICTIONS=False,
    compute_indices=True,
 )

print(f"Evaluation for '{model_type}' model complete.")

In [ ]:
# --- Prepare Kaggle submission ---
import src.utils.kaggle_submission as ks
from src.utils.indices_utils import format_indices
precomputed_indices = False

if not precomputed_indices:
    # Format the indices and save as .nc (optional)
    indices_pred = format_indices(indices_pred, model_type, DATA_VERSION, save_indices=True)
    indices_pred = xr.Dataset(indices_pred)

    if DATA_VERSION == 'lite':
        # Upscale indices to original resolution (x16 in each spatial dimension).
        # IMPORTANT: your target grid (LAT/LON) extends beyond the coarse grid bounds,
        # and xarray fills out-of-bounds with NaN by default even with method='nearest'.
        # Using fill_value='extrapolate' prevents those edge NaNs by repeating boundary values.
        from src.utils.original_data_values import LAT, LON
        lat = np.asarray(LAT, dtype=float)
        lon = np.asarray(LON, dtype=float)

        # Make sure source coords are sorted/monotonic for interpolation
        indices_pred = indices_pred.sortby('lat').sortby('lon')
        indices_pred = indices_pred.interp(
            lat=lat,
            lon=lon,
            method='nearest',
            kwargs={'fill_value': 'extrapolate'},
        )

else:
    # Load the indices 
    indices_pred = xr.open_dataset(f"results_{DATA_VERSION}/{model_type}_indices.nc")

# Save the indices in the required format for Kaggle submission
submission_path = Path(f'submission_{DATA_VERSION}/submission_{model_type}.parquet')
ks.write_kaggle_submission_parquet(indices_pred, submission_path)

# Visualization of results

In [ ]:
# Visualize the results
save_dir = Path(f"results_{DATA_VERSION}/{model_type}_visuals")
emulator.visualize(
    y_pred,
    indices_pred, 
    save_dir,
    variables_to_plot={'y': ['tas', 'tasmax', 'tasmin', 'pr', 'huss', 'psl', 'sfcWind'], 
                       'indices': ['FD', 'SU', 'ID', 'TR', 'GSL', 'CDD', 'CWD', 'SDII', 'R10mm', 'R95p', 'Rx5day', 'TXx', 'TNn', 'WSDI', 'CSDI']},
    time_index=50,
    lat=40.0,
    lon=-95.0
)
print(f"Visualizations saved to {save_dir}")